# 🏆 Visual Anomaly Detection — Part 3: Benchmark & Compare All Models

**Goal**: Run PatchCore, STFPM, and our Custom Autoencoder on multiple MVTec categories and compare results.

**Runtime**: `Runtime > Change runtime type > T4 GPU`

---

### Models Being Compared

| Model | Architecture | Speed | Expected AUROC |
|-------|-------------|-------|---------|
| PatchCore | Memory Bank + k-NN | ⚡ Fast train | ~0.98-1.0 |
| STFPM | Student-Teacher | ⚡ Fast inference | ~0.95-0.99 |
| Custom AE | Convolutional Autoencoder | ☕ Medium | ~0.80-0.95 |

## 1️⃣ Setup

In [ ]:
!pip install -q anomalib torch torchvision opencv-python matplotlib seaborn scikit-learn tqdm pandas tabulate

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import cv2
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
import time

from sklearn.metrics import roc_auc_score, f1_score, roc_curve
from anomalib.data import MVTecAD
from anomalib.models import Patchcore, Stfpm
from anomalib.engine import Engine
from torchvision.transforms.v2 import Resize

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## 2️⃣ Define Custom Autoencoder (same as Notebook 2)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, padding=kernel_size//2, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class AnomalyAutoencoder(nn.Module):
    def __init__(self, in_channels=3, latent_dim=128, encoder_channels=None):
        super().__init__()
        if encoder_channels is None: encoder_channels = [32, 64, 128, 256]
        self.encoder_channels = encoder_channels
        decoder_channels = list(reversed(encoder_channels))

        enc = []
        prev = in_channels
        for ch in encoder_channels:
            enc += [ConvBlock(prev, ch), nn.MaxPool2d(2, 2)]
            prev = ch
        self.encoder = nn.Sequential(*enc)
        self.bottleneck_down = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(encoder_channels[-1], latent_dim), nn.ReLU(inplace=True))
        self.bottleneck_up = nn.Sequential(
            nn.Linear(latent_dim, encoder_channels[-1]*16*16), nn.ReLU(inplace=True))

        dec = []
        prev = decoder_channels[0]
        for ch in decoder_channels[1:]:
            dec.append(DeconvBlock(prev, ch))
            prev = ch
        dec += [DeconvBlock(prev, prev), nn.Conv2d(prev, in_channels, 3, padding=1), nn.Sigmoid()]
        self.decoder = nn.Sequential(*dec)

    def encode(self, x): return self.bottleneck_down(self.encoder(x))
    def decode(self, z):
        x = self.bottleneck_up(z).view(-1, self.encoder_channels[-1], 16, 16)
        return self.decoder(x)
    def forward(self, x):
        latent = self.encode(x)
        recon = self.decode(latent)
        if recon.shape != x.shape:
            recon = F.interpolate(recon, size=x.shape[2:], mode='bilinear', align_corners=False)
        return recon, latent
    def compute_anomaly_score(self, x):
        self.eval()
        with torch.no_grad():
            recon, _ = self(x)
            error = ((x - recon)**2).mean(dim=1)
            return error.view(x.size(0), -1).max(dim=1)[0]

print("✅ Autoencoder defined")

## 3️⃣ Benchmark Functions

In [ ]:
def train_and_eval_anomalib(model_name, model, category, max_epochs=1):
    """Train an Anomalib model and return metrics."""
    data = MVTecAD(
        root='./datasets/MVTecAD', category=category,
        train_batch_size=32, eval_batch_size=32, num_workers=2,
        augmentations=Resize((256, 256)),
    )
    engine = Engine(
        max_epochs=max_epochs, accelerator='auto', devices=1,
        default_root_dir=f'./results/{model_name}/{category}',
    )

    t0 = time.time()
    engine.fit(model=model, datamodule=data)
    train_time = time.time() - t0

    results = engine.test(model=model, datamodule=data)

    metrics = {'model': model_name, 'category': category, 'training_time_s': round(train_time, 1)}
    for d in results:
        for k, v in d.items():
            if isinstance(v, float):
                metrics[k] = round(v, 4)
    return metrics


def train_and_eval_autoencoder(category, epochs=50):
    """Train custom autoencoder and return metrics."""
    data = MVTecAD(
        root='./datasets/MVTecAD', category=category,
        train_batch_size=16, eval_batch_size=16, num_workers=2,
        augmentations=Resize((256, 256)),
    )
    data.setup()
    train_loader = data.train_dataloader()
    test_loader = data.test_dataloader()

    model = AnomalyAutoencoder().to(device)
    optimizer = Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.MSELoss()

    t0 = time.time()
    best_loss = float('inf')
    patience = 10
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            images = batch['image'].to(device)
            optimizer.zero_grad()
            recon, _ = model(images)
            loss = criterion(recon, images)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Val loss
        model.eval()
        vl = 0; vn = 0
        with torch.no_grad():
            for batch in test_loader:
                imgs = batch['image'].to(device)
                mask = batch['label'] == 0
                if mask.sum() == 0: continue
                r, _ = model(imgs[mask])
                vl += criterion(r, imgs[mask]).item(); vn += 1
        avg_vl = vl / max(vn, 1)
        if avg_vl < best_loss:
            best_loss = avg_vl; patience_counter = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1
            if patience_counter >= patience: break

    train_time = time.time() - t0
    model.load_state_dict(best_state)

    # Evaluate
    model.eval()
    all_scores, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(device)
            scores = model.compute_anomaly_score(images)
            all_scores.extend(scores.cpu().numpy().tolist())
            labels = batch['label']
            all_labels.extend(labels.numpy().tolist() if hasattr(labels, 'numpy') else labels)

    auroc = roc_auc_score(all_labels, all_scores)
    return {
        'model': 'Custom AE', 'category': category,
        'image_AUROC': round(auroc, 4),
        'training_time_s': round(train_time, 1),
    }

print("✅ Benchmark functions ready")

## 4️⃣ Run the Benchmark!

We'll run all 3 models on 3 representative categories:
- **bottle** — rigid object with clear defects
- **carpet** — texture with subtle anomalies
- **screw** — small defects on tiny object

In [ ]:
# ⚙️ Benchmark config
CATEGORIES = ["bottle", "carpet", "screw"]  # Add more if you want (takes more time)

all_results = []

for cat in CATEGORIES:
    print(f"\n{'='*60}")
    print(f"  Category: {cat.upper()}")
    print(f"{'='*60}")

    # PatchCore
    print(f"\n  🔵 Training PatchCore on {cat}...")
    pc_model = Patchcore(backbone='wide_resnet50_2', layers=('layer2', 'layer3'),
                         coreset_sampling_ratio=0.1, num_neighbors=9)
    pc_result = train_and_eval_anomalib('PatchCore', pc_model, cat, max_epochs=1)
    all_results.append(pc_result)
    print(f"     → AUROC: {pc_result.get('image_AUROC', 'N/A')} ({pc_result['training_time_s']}s)")

    # STFPM
    print(f"\n  🟢 Training STFPM on {cat}...")
    st_model = Stfpm(backbone='resnet18', layers=('layer1', 'layer2', 'layer3'))
    st_result = train_and_eval_anomalib('STFPM', st_model, cat, max_epochs=50)
    all_results.append(st_result)
    print(f"     → AUROC: {st_result.get('image_AUROC', 'N/A')} ({st_result['training_time_s']}s)")

    # Custom Autoencoder
    print(f"\n  🟡 Training Custom AE on {cat}...")
    ae_result = train_and_eval_autoencoder(cat, epochs=50)
    all_results.append(ae_result)
    print(f"     → AUROC: {ae_result.get('image_AUROC', 'N/A')} ({ae_result['training_time_s']}s)")

print("\n✅ Benchmark complete!")

In [ ]:
# Create results DataFrame
df = pd.DataFrame(all_results)
print("\n🏆 BENCHMARK RESULTS")
print("=" * 70)
print(df.to_string(index=False))

## 5️⃣ Visualize Results

In [ ]:
# Grouped bar chart: AUROC by model and category
if 'image_AUROC' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Bar chart
    pivot = df.pivot(index='category', columns='model', values='image_AUROC')
    colors = ['#2196F3', '#4CAF50', '#FF9800']
    pivot.plot(kind='bar', ax=axes[0], color=colors[:len(pivot.columns)],
               edgecolor='white', linewidth=2)
    axes[0].set_ylabel('Image AUROC', fontsize=12)
    axes[0].set_title('Model Comparison by Category', fontweight='bold', fontsize=14)
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
    axes[0].set_ylim(0.5, 1.05)
    axes[0].grid(axis='y', alpha=0.3)
    axes[0].legend(title='Model')

    # Add value labels on bars
    for container in axes[0].containers:
        axes[0].bar_label(container, fmt='%.3f', fontsize=8, fontweight='bold')

    # Training time comparison
    pivot_time = df.pivot(index='category', columns='model', values='training_time_s')
    pivot_time.plot(kind='bar', ax=axes[1], color=colors[:len(pivot_time.columns)],
                    edgecolor='white', linewidth=2)
    axes[1].set_ylabel('Training Time (seconds)', fontsize=12)
    axes[1].set_title('Training Time by Category', fontweight='bold', fontsize=14)
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    axes[1].grid(axis='y', alpha=0.3)
    axes[1].legend(title='Model')

    plt.tight_layout()
    plt.show()
else:
    print('image_AUROC column not found. Check your metric names:')
    print(df.columns.tolist())

In [ ]:
# Summary stats
if 'image_AUROC' in df.columns:
    summary = df.groupby('model').agg(
        mean_auroc=('image_AUROC', 'mean'),
        std_auroc=('image_AUROC', 'std'),
        min_auroc=('image_AUROC', 'min'),
        max_auroc=('image_AUROC', 'max'),
        total_time=('training_time_s', 'sum'),
    ).round(4).sort_values('mean_auroc', ascending=False)

    print("\n🏆 LEADERBOARD")
    print("=" * 70)
    print(summary.to_string())

    # Winner
    winner = summary.index[0]
    print(f"\n🥇 Winner: {winner} (mean AUROC = {summary.loc[winner, 'mean_auroc']:.4f})")

In [ ]:
# Heatmap of AUROC scores
if 'image_AUROC' in df.columns:
    pivot = df.pivot(index='category', columns='model', values='image_AUROC')

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn',
                vmin=0.7, vmax=1.0, linewidths=2, linecolor='white',
                cbar_kws={'label': 'Image AUROC'}, ax=ax)
    ax.set_title('AUROC Heatmap: Model \u00d7 Category', fontweight='bold', fontsize=14)
    ax.set_ylabel('Category')
    ax.set_xlabel('Model')
    plt.tight_layout()
    plt.show()

## 6️⃣ Analysis & Key Takeaways

### Expected Results

| Model | Typical AUROC | Why |
|-------|-------------|-----|
| **PatchCore** | 0.98–1.00 | Pretrained ImageNet features + optimal k-NN. SOTA baseline. |
| **STFPM** | 0.95–0.99 | Strong multi-scale distillation, fast inference |
| **Custom AE** | 0.80–0.95 | Simpler architecture, but demonstrates the core concept |

### Why PatchCore Typically Wins
1. **Transfer learning**: Uses frozen WideResNet-50 pretrained on ImageNet (huge head start)
2. **No training required**: Just stores features — no optimization pitfalls
3. **Coreset sampling**: Memory-efficient representation of the normal distribution

### Why Custom AE Still Matters
1. **Understand the fundamentals**: Reconstruction-based detection is the foundation
2. **Full control**: You built every component from scratch
3. **Interview gold**: Explaining WHY PatchCore beats AE shows deep understanding

---

### 🎓 Interview-Ready Summary

| Question | Expert Answer |
|----------|-------------|
| Compare PatchCore vs Autoencoder | PatchCore uses pretrained features + k-NN (no training needed), AE learns from scratch (harder). PatchCore wins on AUROC but AE teaches fundamentals. |
| What is the accuracy-speed tradeoff? | PatchCore: fast train, slower inference (k-NN lookup). STFPM: medium train, fastest inference. AE: slowest train, medium inference. |
| How would you improve the AE? | Use perceptual loss (VGG features), attention mechanisms, or combine with pretrained encoder (hybrid approach). |
| What makes MVTec AD challenging? | Some categories have very subtle defects (e.g., screw thread damage). Texture categories need different features than objects. |
| How to deploy in production? | PatchCore for accuracy, EfficientAD for edge/real-time. Export to ONNX/OpenVINO. Set threshold based on acceptable false positive rate. |